<a href="https://colab.research.google.com/github/smsag99/Thesis/blob/main/codes/Hierchical_Structure.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IMPORTANT :



In [1]:
path = '/content/drive/MyDrive/Thesis_Data/'

## preprocessing Data (removing outliers)

In [4]:
def remove_outliers(df_animal):
  df_animal_filtered = df_animal[(df_animal['milk_kg'] >= 0.4) & (df_animal['milk_kg'] <= 26.5)]
  print(f"Total rows removed after milk filtering: {len(df_animal)-len(df_animal_filtered)}")
  df_animal_filtered = df_animal_filtered[(df_animal_filtered['fat_p'] >= 1.6) & (df_animal_filtered['fat_p'] <= 15.05)]
  print(f"Total rows removed after fat filtering: {len(df_animal)-len(df_animal_filtered)}")
  df_animal_filtered = df_animal_filtered[(df_animal_filtered['protein_p'] >= 2.67) & (df_animal_filtered['protein_p'] <= 6.68)]
  print(f"Total rows removed after protein filtering: {len(df_animal)-len(df_animal_filtered)}")
  df_animal_filtered = df_animal_filtered[(df_animal_filtered['SCS'] >= -2.06) & (df_animal_filtered['SCS'] <= 10.73)]
  print(f"Total rows removed after SCS filtering: {len(df_animal)-len(df_animal_filtered)}")
  df_animal_filtered = df_animal_filtered[df_animal_filtered['ECM'] <= 301]
  print(f"Total rows removed after ECM filtering: {len(df_animal)-len(df_animal_filtered)}")
  df_animal_filtered['AFC'] = pd.to_timedelta(df_animal_filtered['AFC'])
  df_animal_filtered = df_animal_filtered[(df_animal_filtered['AFC'] >= pd.Timedelta(days=720)) & (df_animal_filtered['AFC'] <= pd.Timedelta(days=1440))]
  print(f"Total rows removed after AFC filtering: {len(df_animal)-len(df_animal_filtered)}")
  print(f"Total rows after filtering: {len(df_animal_filtered)}")
  return df_animal_filtered

df_Merged_Scaled_Processed = remove_outliers(df_Merged_Scaled_uprocessed)
df_Merged_Scaled_Processed.to_csv(path + '/Final_Data/Merged_Data_Scaled_Processed.csv', index=False)
print("saved")

NameError: name 'df_Merged_Scaled_uprocessed' is not defined

In [2]:
def convert_types(df):
  df['dtb'] = pd.to_datetime(df['dtb'])
  df['dtt'] = pd.to_datetime(df['dtt'])
  df['dtc'] = pd.to_datetime(df['dtc'])
  df['AFC'] = pd.to_timedelta(df['AFC']).dt.days
  df['DIM'] = pd.to_timedelta(df['DIM']).dt.days
  return df

## Claude Method

### Step 1 — Build the summing matrix S
#### This is the core of HTS. The S matrix maps bottom-level series (Animal × Parity) to every level above.


In [29]:
import pandas as pd
import numpy as np

df = pd.read_csv(path + '/Final_Data/Merged_Data_Scaled_Processed.csv')
df = convert_types(df)
df['dtt'] = pd.to_datetime(df['dtt'])

# Create a unique bottom-level key: Animal_Parity
df['series_key'] = df['Animal_ID'] + '_P' + df['parity'].astype(str)

# Pivot: each column = one bottom-level series, rows = dates
bottom = df.pivot_table(index='dtt', columns='series_key', values='milk_kg', aggfunc='sum').fillna(0)

# Animal-level aggregates (sum parities per animal per date)
animal_level = df.pivot_table(index='dtt', columns='Animal_ID', values='milk_kg', aggfunc='sum').fillna(0)

# Farm-level aggregate
farm_level = df.pivot_table(index='dtt', columns='Farm_Code', values='milk_kg', aggfunc='sum').fillna(0)

### Step 2 — Use the hts or statsforecast library
#### The best Python option today is hierarchicalforecast from Nixtla:

In [31]:
%%capture
pip install hierarchicalforecast statsforecast

In [32]:
from hierarchicalforecast.core import HierarchicalReconciliation
from hierarchicalforecast.methods import BottomUp, MinTrace
from statsforecast import StatsForecast
from statsforecast.models import AutoETS

# Define hierarchy as a list-of-lists: [farm, animal, series_key]
# Each row of df needs all three levels filled
df['Farm'] = df['Farm_Code'].astype(str)
df['Animal'] = df['Animal_ID']
df['Parity'] = 'P' + df['parity'].astype(str)

# Build the Y_df in Nixtla format: unique_id, ds, y
Y_df = df[['series_key', 'dtt', 'milk_kg']].rename(columns={
    'series_key': 'unique_id', 'dtt': 'ds', 'milk_kg': 'y'
})

# Also add upper-level series manually
animal_df = df.groupby(['Animal_ID', 'dtt'])['milk_kg'].sum().reset_index()
animal_df['unique_id'] = animal_df['Animal_ID']
animal_df = animal_df.rename(columns={'dtt': 'ds', 'milk_kg': 'y'})

farm_df = df.groupby(['Farm_Code', 'dtt'])['milk_kg'].sum().reset_index()
farm_df['unique_id'] = farm_df['Farm_Code'].astype(str)
farm_df = farm_df.rename(columns={'dtt': 'ds', 'milk_kg': 'y'})

# Combine all levels
Y_df_all = pd.concat([
    farm_df[['unique_id','ds','y']],
    animal_df[['unique_id','ds','y']],
    Y_df
]).sort_values(['unique_id', 'ds'])

### Step 3 — Define the hierarchy tags

In [33]:
# S_df: summing matrix (columns = bottom series, rows = all series)
# tags: dict mapping each level name to the list of series at that level

tags = {
    'Farm':   farm_df['unique_id'].unique().tolist(),
    'Animal': animal_df['unique_id'].unique().tolist(),
    'Parity': df['series_key'].unique().tolist()
}

### Step 4 — Fit base forecasters and reconcile

In [34]:
from statsforecast import StatsForecast
from statsforecast.models import AutoETS

# Fit one model per bottom-level series
sf = StatsForecast(models=[AutoETS(season_length=12)], freq='MS', n_jobs=-1)
sf.fit(Y_df_all)
forecasts_df = sf.predict(h=6)  # 6 periods ahead

# Reconcile: MinTrace is the gold standard
hrec = HierarchicalReconciliation(reconcilers=[
    BottomUp(),
    MinTrace(method='mint_shrink')
])

reconciled = hrec.reconcile(
    Y_hat_df=forecasts_df,
    Y_df=Y_df_all,
    tags=tags
)



KeyError: 'fitted'

## Gemini Method


### Step 1: Preprocess the Data

In [3]:
import pandas as pd

# 1. Load and parse dates
df = pd.read_csv(path + '/Final_Data/Merged_Data_Scaled_Processed.csv')
df = convert_types(df)
df['dtt'] = pd.to_datetime(df['dtt'])

# 2. Select relevant columns
ts_df = df[['Farm_Code', 'Animal_ID', 'dtt', 'milk_kg']].copy()

# 3. Create a unique identifier for the bottom level
ts_df['Animal_Key'] = ts_df['Farm_Code'].astype(str) + "_" + ts_df['Animal_ID'].astype(str)
ts_df = ts_df.rename(columns={'dtt': 'ds', 'milk_kg': 'y'})

# 4. Resample to a regular frequency (e.g., Monthly 'ME')
ts_df = ts_df.set_index('ds').groupby('Animal_Key').resample('ME')['y'].sum().reset_index()

In [4]:
# 5. SAVE TO CSV
ts_df.to_csv('preprocessed_hts_data.csv', index=False)
print("Step 1 complete. Data saved to preprocessed_hts_data.csv")

Step 1 complete. Data saved to preprocessed_hts_data.csv


### Step 2: Build the Hierarchy aggregation

In [7]:
%%capture
!pip install hierarchicalforecast statsforecast

In [ ]:
import pandas as pd
import numpy as np

# 1. Load data
ts_df = pd.read_csv('preprocessed_hts_data.csv', parse_dates=['ds'])
ts_df['Farm'] = ts_df['Animal_Key'].apply(lambda x: x.split('_')[0])

print("Aggregating Bottom Level...")
# Bottom Level (Animal)
bottom_df = ts_df[['Animal_Key', 'ds', 'y']].rename(columns={'Animal_Key': 'unique_id'})

print("Aggregating Farm Level...")
# Level 1 (Farm)
farm_df = ts_df.groupby(['Farm', 'ds'])['y'].sum().reset_index()
farm_df = farm_df.rename(columns={'Farm': 'unique_id'})

print("Aggregating Total Level...")
# Level 0 (Total)
total_df = ts_df.groupby('ds')['y'].sum().reset_index()
total_df['unique_id'] = 'Total'

print("Combining Y_df...")
# Combine all levels into Y_df
Y_df = pd.concat([total_df, farm_df, bottom_df], ignore_index=True)

# Free up memory
del ts_df, total_df, farm_df, bottom_df
import gc
gc.collect()

print("Building tags and S_df...")

# 2. Build the tags dictionary manually
unique_farms = Y_df[Y_df['unique_id'].str.contains('_') == False]['unique_id'].unique()
unique_farms = [f for f in unique_farms if f != 'Total']
unique_animals = Y_df[Y_df['unique_id'].str.contains('_')]['unique_id'].unique()

tags = {
    'Farm': np.array(unique_farms),
    'Farm/Animal_Key': np.array(unique_animals)
}

# 3. Build S_df (Summing Matrix) dynamically and memory-efficiently
# Using uint8 (1 byte per cell) instead of float64 (8 bytes per cell) saves 87% of memory!
S_dict = {}

# Create a mapping DataFrame
mapping = pd.DataFrame({'Animal_Key': unique_animals})
mapping['Farm'] = mapping['Animal_Key'].apply(lambda x: x.split('_')[0])

for animal in unique_animals:
    # Get the farm for this animal
    farm = animal.split('_')[0]

    # Define the 1s and 0s for this specific animal column
    col = pd.Series(0, index=['Total'] + list(unique_farms) + list(unique_animals), dtype=np.uint8)

    col['Total'] = 1
    col[farm] = 1
    col[animal] = 1

    S_dict[animal] = col

# Convert dictionary to DataFrame
S_df = pd.DataFrame(S_dict)

print("Step 2 completed without running out of RAM!")

Aggregating Bottom Level...
Aggregating Farm Level...
Aggregating Total Level...
Combining Y_df...
Building tags and S_df...


### Step 3: Train Base Models

In [ ]:
from statsforecast.models import AutoARIMA, ETS
from statsforecast.core import StatsForecast

# Define base models
models = [
    AutoARIMA(season_length=12),
    ETS(season_length=12, model='ZZA')
]

# Generate base forecasts (e.g., forecast next 6 months)
sf = StatsForecast(df=Y_df, models=models, freq='M', n_jobs=-1)
Y_hat_df = sf.forecast(h=6)

### Step 4: Reconcile the Forecasts

In [ ]:
from hierarchicalforecast.methods import BottomUp, TopDown, MinT
from hierarchicalforecast.core import HierarchicalForecast

# Choose reconciliation methods
reconcilers = [
    BottomUp(),             # Adds bottom level up to the top
    TopDown(method='forecast_proportions'), # Distributes top level down to bottom
    MinT(method='wls_var')  # Optimal combination (usually the most accurate)
]

# Apply reconciliation
hforecast = HierarchicalForecast(models=reconcilers, freq='M')
Y_rec_df = hforecast.reconcile(Y_hat_df=Y_hat_df, Y_df=Y_df, S=S_df, tags=tags)